# Phase 2: Temporal & Graph-Based Feature Engineering



## 1. Load validated temporal snapshot inputs

Loads five exported tables and normalizes checkpoint timestamps.

In [ ]:
# ============================================================
# NOTEBOOK 2 — STEP 1.1
# LOAD FINAL TEMPORAL SNAPSHOT DATA
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/GC1")

SNAPSHOT_DIR = (
    ROOT / "Temporal_snapshots" / "final_snapshots"
)

assert SNAPSHOT_DIR.exists(), (
    f"Snapshot directory not found: {SNAPSHOT_DIR}"
)

# ------------------------------------------------------------
# Load Notebook 1 outputs
# ------------------------------------------------------------

nodes_df = pd.read_parquet(
    SNAPSHOT_DIR / "nodes.parquet"
)

edges_df = pd.read_parquet(
    SNAPSHOT_DIR / "edges.parquet"
)

jira_states_df = pd.read_parquet(
    SNAPSHOT_DIR / "jira_states.parquet"
)

dependencies_df = pd.read_parquet(
    SNAPSHOT_DIR / "active_dependencies.parquet"
)

checkpoints_df = pd.read_parquet(
    SNAPSHOT_DIR / "snapshot_manifest.parquet"
)

# ------------------------------------------------------------
# Standardize timestamps
# ------------------------------------------------------------

timestamp_columns = {
    "nodes": (nodes_df, ["available_at"]),
    "edges": (edges_df, ["available_at"]),
    "jira": (
        jira_states_df,
        ["checkpoint_time", "latest_status_change"]
    ),
    "dependencies": (
        dependencies_df,
        ["checkpoint_time", "active_from"]
    ),
    "checkpoints": (
        checkpoints_df,
        ["checkpoint_time"]
    )
}

for name, (df, columns) in timestamp_columns.items():

    for col in columns:

        if col in df.columns:

            df[col] = pd.to_datetime(
                df[col],
                utc=True,
                errors="coerce",
                format="mixed"
            )

# ------------------------------------------------------------
# Display input summary
# ------------------------------------------------------------

summary = pd.DataFrame([
    {
        "dataset": "Temporal KG Nodes",
        "records": len(nodes_df),
        "columns": len(nodes_df.columns)
    },
    {
        "dataset": "Temporal KG Edges",
        "records": len(edges_df),
        "columns": len(edges_df.columns)
    },
    {
        "dataset": "Historical Jira States",
        "records": len(jira_states_df),
        "columns": len(jira_states_df.columns)
    },
    {
        "dataset": "Active Dependencies",
        "records": len(dependencies_df),
        "columns": len(dependencies_df.columns)
    },
    {
        "dataset": "Weekly Checkpoints",
        "records": len(checkpoints_df),
        "columns": len(checkpoints_df.columns)
    }
])

print("=" * 65)
print("NOTEBOOK 2 — INPUT DATA SUMMARY")
print("=" * 65)

display(summary)

print("\nProjects:",
      checkpoints_df["instance_id"].nunique())

print("Checkpoints:", len(checkpoints_df))

print("\nCheckpoint sample:")
display(checkpoints_df.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
NOTEBOOK 2 — INPUT DATA SUMMARY


,dataset,records,columns
0,Temporal KG Nodes,96487,8
1,Temporal KG Edges,164101,6
2,Historical Jira States,228255,6
3,Active Dependencies,12684,7
4,Weekly Checkpoints,987,10



Projects: 30
Checkpoints: 987

Checkpoint sample:


,snapshot_id,instance_id,week_number,checkpoint_time,kg_nodes,kg_edges,historical_issues,predictive_dependency_links,predictive_blocked_issues,predictive_upstream_issues
0,AEROGEAR-I01::W01,AEROGEAR-I01,1,2018-01-24 05:30:01+00:00,239,142,162,0,0,0
1,AEROGEAR-I01::W02,AEROGEAR-I01,2,2018-01-31 05:30:01+00:00,271,202,162,0,0,0
2,AEROGEAR-I01::W03,AEROGEAR-I01,3,2018-02-07 05:30:01+00:00,284,228,162,0,0,0
3,AEROGEAR-I01::W04,AEROGEAR-I01,4,2018-02-14 05:30:01+00:00,328,308,164,0,0,0
4,AEROGEAR-I01::W05,AEROGEAR-I01,5,2018-02-21 05:30:01+00:00,365,374,166,2,1,2


## 2. Historical Jira workflow features

Reconstructs status indicators and recent status-change features.

In [ ]:
# ============================================================
# NOTEBOOK 2 — STEP 2.1
# DEFINE HISTORICAL JIRA STATUS CATEGORIES
# ============================================================

import pandas as pd
import numpy as np

jira_workflow = jira_states_df.copy()

jira_workflow["status_normalized"] = (
    jira_workflow["historical_status"]
    .fillna("UNKNOWN")
    .astype(str)
    .str.strip()
    .str.lower()
)

# Inspect actual status distribution
print("HISTORICAL STATUS DISTRIBUTION")

display(
    jira_workflow["status_normalized"]
    .value_counts()
    .head(40)
)

# Conservative terminal status mapping
TERMINAL_STATUSES = {
    "closed",
    "resolved",
    "done",
    "verified",
    "completed",
    "cancelled",
    "canceled",
    "won't fix",
    "wont fix"
}

jira_workflow["is_terminal"] = (
    jira_workflow["status_normalized"]
    .isin(TERMINAL_STATUSES)
)

jira_workflow["is_unknown"] = (
    jira_workflow["status_normalized"] == "unknown"
)

jira_workflow["is_open"] = (
    ~jira_workflow["is_terminal"]
    & ~jira_workflow["is_unknown"]
)

jira_workflow["is_reopened"] = (
    jira_workflow["status_normalized"] == "reopened"
)

jira_workflow["is_blocked"] = (
    jira_workflow["status_normalized"] == "blocked"
)

print("\nSTATUS CLASSIFICATION")

print("Terminal records:", jira_workflow["is_terminal"].sum())
print("Open records:", jira_workflow["is_open"].sum())
print("Unknown records:", jira_workflow["is_unknown"].sum())

HISTORICAL STATUS DISTRIBUTION


,count
status_normalized,
closed,118405
resolved,50582
open,17810
unknown,12727
reopened,8133
gathering interest,3677
in progress,2850
done,2485
verified,1678



STATUS CLASSIFICATION
Terminal records: 173150
Open records: 42378
Unknown records: 12727


In [ ]:
# ============================================================
# NOTEBOOK 2 — STEP 2.2
# RECENT HISTORICAL STATUS CHANGE INDICATORS
# ============================================================

jira_workflow["latest_status_change"] = pd.to_datetime(
    jira_workflow["latest_status_change"],
    utc=True,
    errors="coerce",
    format="mixed"
)

jira_workflow["days_since_status_change"] = (
    jira_workflow["checkpoint_time"]
    - jira_workflow["latest_status_change"]
).dt.total_seconds() / 86400

jira_workflow["changed_last_7d"] = (
    jira_workflow["days_since_status_change"]
    .ge(0)
    &
    jira_workflow["days_since_status_change"]
    .le(7)
)

jira_workflow["changed_last_14d"] = (
    jira_workflow["days_since_status_change"]
    .ge(0)
    &
    jira_workflow["days_since_status_change"]
    .le(14)
)

print("Recent status indicators prepared.")

display(
    jira_workflow[
        [
            "instance_id",
            "week_number",
            "issue_key",
            "historical_status",
            "days_since_status_change",
            "changed_last_7d",
            "changed_last_14d"
        ]
    ].head(10)
)

Recent status indicators prepared.


,instance_id,week_number,issue_key,historical_status,days_since_status_change,changed_last_7d,changed_last_14d
0,AEROGEAR-I01,1,AEROGEAR-7517,Resolved,1651.978067,False,False
1,AEROGEAR-I01,1,AEROGEAR-7491,Closed,1618.814306,False,False
2,AEROGEAR-I01,1,AEROGEAR-7474,Closed,1624.139965,False,False
3,AEROGEAR-I01,1,AEROGEAR-7390,Closed,253.031215,False,False
4,AEROGEAR-I01,1,AEROGEAR-7383,Closed,1481.864884,False,False
5,AEROGEAR-I01,1,AEROGEAR-7344,Closed,1434.860475,False,False
6,AEROGEAR-I01,1,AEROGEAR-7256,Resolved,1443.040764,False,False
7,AEROGEAR-I01,1,AEROGEAR-7200,Closed,1434.981343,False,False
8,AEROGEAR-I01,1,AEROGEAR-7184,Closed,418.052535,False,False
9,AEROGEAR-I01,1,AEROGEAR-7159,Closed,1434.981412,False,False


In [ ]:
# ============================================================
# NOTEBOOK 2 — STEP 2.3
# AGGREGATE HISTORICAL WORKFLOW FEATURES
# ============================================================

KEYS = [
    "instance_id",
    "week_number",
    "checkpoint_time"
]

workflow_features = (
    jira_workflow.groupby(KEYS)
    .agg(
        total_issues=("issue_key", "nunique"),
        open_issues=("is_open", "sum"),
        reopened_issues=("is_reopened", "sum"),
        blocked_status_issues=("is_blocked", "sum"),
        unknown_status_issues=("is_unknown", "sum"),
        recent_status_changes_7d=("changed_last_7d", "sum"),
        recent_status_changes_14d=("changed_last_14d", "sum")
    )
    .reset_index()
)

workflow_features["open_issue_ratio"] = (
    workflow_features["open_issues"]
    / workflow_features["total_issues"].clip(lower=1)
)

workflow_features["unknown_status_ratio"] = (
    workflow_features["unknown_status_issues"]
    / workflow_features["total_issues"].clip(lower=1)
)

workflow_features["status_change_ratio_14d"] = (
    workflow_features["recent_status_changes_14d"]
    / workflow_features["total_issues"].clip(lower=1)
)

# Preserve all 987 checkpoints
workflow_features = checkpoints_df[KEYS].merge(
    workflow_features,
    on=KEYS,
    how="left",
    validate="one_to_one"
)

print("Workflow feature records:", len(workflow_features))

display(workflow_features.head(10))

print("\nFEATURE DISTRIBUTION")

display(
    workflow_features.drop(
        columns=KEYS
    ).describe()
)

Workflow feature records: 987


,instance_id,week_number,checkpoint_time,total_issues,open_issues,reopened_issues,blocked_status_issues,unknown_status_issues,recent_status_changes_7d,recent_status_changes_14d,open_issue_ratio,unknown_status_ratio,status_change_ratio_14d
0,AEROGEAR-I01,1,2018-01-24 05:30:01+00:00,162,4,2,0,14,2,2,0.024691,0.086420,0.012346
1,AEROGEAR-I01,2,2018-01-31 05:30:01+00:00,162,7,2,0,11,3,5,0.043210,0.067901,0.030864
2,AEROGEAR-I01,3,2018-02-07 05:30:01+00:00,162,9,2,0,9,2,5,0.055556,0.055556,0.030864
3,AEROGEAR-I01,4,2018-02-14 05:30:01+00:00,164,7,2,0,11,3,3,0.042683,0.067073,0.018293
4,AEROGEAR-I01,5,2018-02-21 05:30:01+00:00,166,6,2,0,11,6,8,0.036145,0.066265,0.048193
5,AEROGEAR-I01,6,2018-02-28 05:30:01+00:00,168,8,2,0,11,3,8,0.047619,0.065476,0.047619
6,AEROGEAR-I01,7,2018-03-07 05:30:01+00:00,177,9,2,0,17,6,7,0.050847,0.096045,0.039548
7,AEROGEAR-I01,8,2018-03-14 05:30:01+00:00,180,8,2,0,12,15,17,0.044444,0.066667,0.094444
8,AEROGEAR-I01,9,2018-03-21 05:30:01+00:00,180,6,2,0,11,5,16,0.033333,0.061111,0.088889
9,AEROGEAR-I01,10,2018-03-28 05:30:01+00:00,180,5,2,0,11,3,6,0.027778,0.061111,0.033333



FEATURE DISTRIBUTION


,total_issues,open_issues,reopened_issues,blocked_status_issues,unknown_status_issues,recent_status_changes_7d,recent_status_changes_14d,open_issue_ratio,unknown_status_ratio,status_change_ratio_14d
count,987.000000,987.00000,987.000000,987.000000,987.000000,987.000000,987.000000,987.000000,987.000000,987.000000
mean,231.261398,42.93617,8.240122,0.595745,12.894630,5.896657,10.002026,0.184203,0.059497,0.047878
std,47.059784,26.75539,17.892911,2.786832,13.355608,12.357318,17.505722,0.105758,0.058428,0.080365
min,46.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,199.000000,25.00000,0.000000,0.000000,3.000000,1.000000,3.000000,0.114943,0.013393,0.013100
50%,244.000000,39.00000,0.000000,0.000000,12.000000,3.000000,6.000000,0.170507,0.047809,0.024896
75%,268.000000,53.00000,7.000000,0.000000,18.000000,6.000000,10.000000,0.217857,0.086957,0.047619
max,299.000000,130.00000,93.000000,20.000000,107.000000,242.000000,243.000000,0.511811,0.405303,0.952941


## 2.2 Identify concentrated workflow changes

Flags candidate bulk-update days and calculates adjusted workflow activity.

In [ ]:
# ============================================================
# NOTEBOOK 2 — STEP 2.6
# IDENTIFY SUSPICIOUS BULK WORKFLOW UPDATES
# ============================================================

import pandas as pd
import numpy as np

MIN_CHANGED_ISSUES = 20
MIN_PROJECT_SHARE = 0.30

workflow_audit = jira_workflow.copy()

workflow_audit["status_change_day"] = (
    workflow_audit["latest_status_change"].dt.floor("D")
)

# Only changes observed within the preceding 14 days
recent = workflow_audit[
    workflow_audit["changed_last_14d"]
].copy()

KEYS = [
    "instance_id",
    "week_number",
    "checkpoint_time"
]

daily_counts = (
    recent.groupby(
        KEYS + ["status_change_day"]
    )
    .agg(
        changed_issues=("issue_key", "nunique")
    )
    .reset_index()
)

daily_counts = daily_counts.merge(
    workflow_features[
        KEYS + ["total_issues"]
    ],
    on=KEYS,
    how="left",
    validate="many_to_one"
)

daily_counts["affected_share"] = (
    daily_counts["changed_issues"]
    / daily_counts["total_issues"]
)

daily_counts["suspected_bulk_day"] = (
    (daily_counts["changed_issues"] >= MIN_CHANGED_ISSUES)
    &
    (daily_counts["affected_share"] >= MIN_PROJECT_SHARE)
)

print("SUSPICIOUS WORKFLOW UPDATE DAYS")

suspected_days = daily_counts[
    daily_counts["suspected_bulk_day"]
].copy()

display(
    suspected_days.sort_values(
        "changed_issues",
        ascending=False
    ).head(30)
)

print("\nCandidate project-days:", len(suspected_days))

print("Projects affected:",
      suspected_days["instance_id"].nunique())

SUSPICIOUS WORKFLOW UPDATE DAYS


,instance_id,week_number,checkpoint_time,status_change_day,changed_issues,total_issues,affected_share,suspected_bulk_day
2468,JSDCLOUD-I01,25,2019-09-06 00:36:22+00:00,2019-08-26 00:00:00+00:00,240,255,0.941176,True
2465,JSDCLOUD-I01,24,2019-08-30 00:36:22+00:00,2019-08-26 00:00:00+00:00,240,254,0.944882,True
2334,JRACLOUD-I01,22,2019-08-22 00:14:02+00:00,2019-08-15 00:00:00+00:00,134,279,0.480287,True
2337,JRACLOUD-I01,23,2019-08-29 00:14:02+00:00,2019-08-15 00:00:00+00:00,133,279,0.476703,True
2377,JRASERVER-I01,3,2018-02-09 00:22:27+00:00,2018-02-08 00:00:00+00:00,121,279,0.433692,True
2378,JRASERVER-I01,4,2018-02-16 00:22:27+00:00,2018-02-08 00:00:00+00:00,118,279,0.422939,True
2268,JRACLOUD-I01,4,2019-04-18 00:14:02+00:00,2019-04-16 00:00:00+00:00,99,266,0.372180,True
2270,JRACLOUD-I01,5,2019-04-25 00:14:02+00:00,2019-04-16 00:00:00+00:00,97,266,0.364662,True
2502,MC-I01,2,2020-04-15 05:07:08+00:00,2020-04-13 00:00:00+00:00,81,239,0.338912,True



Candidate project-days: 9
Projects affected: 4


In [ ]:
# ============================================================
# NOTEBOOK 2 — STEP 2.7
# BULK-ADJUSTED WORKFLOW FEATURES
# ============================================================

bulk_day_keys = suspected_days[
    KEYS + ["status_change_day"]
].drop_duplicates().copy()

bulk_day_keys["is_suspected_bulk"] = True

workflow_adjusted = workflow_audit.merge(
    bulk_day_keys,
    on=KEYS + ["status_change_day"],
    how="left",
    validate="many_to_one"
)

workflow_adjusted["is_suspected_bulk"] = (
    workflow_adjusted["is_suspected_bulk"].eq(True)
)

workflow_adjusted["adjusted_change_7d"] = (
    workflow_adjusted["changed_last_7d"]
    &
    ~workflow_adjusted["is_suspected_bulk"]
)

workflow_adjusted["adjusted_change_14d"] = (
    workflow_adjusted["changed_last_14d"]
    &
    ~workflow_adjusted["is_suspected_bulk"]
)

adjusted_counts = (
    workflow_adjusted.groupby(KEYS)
    .agg(
        recent_status_changes_7d_bulk_adjusted=(
            "adjusted_change_7d", "sum"
        ),
        recent_status_changes_14d_bulk_adjusted=(
            "adjusted_change_14d", "sum"
        ),
        suspected_bulk_issues_14d=(
            "is_suspected_bulk", "sum"
        )
    )
    .reset_index()
)

workflow_features = workflow_features.merge(
    adjusted_counts,
    on=KEYS,
    how="left",
    validate="one_to_one"
)

workflow_features["status_change_ratio_14d_bulk_adjusted"] = (
    workflow_features[
        "recent_status_changes_14d_bulk_adjusted"
    ]
    / workflow_features["total_issues"].clip(lower=1)
)

print("UPDATED WORKFLOW FEATURES")

display(
    workflow_features[
        [
            "instance_id",
            "week_number",
            "recent_status_changes_7d",
            "recent_status_changes_7d_bulk_adjusted",
            "recent_status_changes_14d",
            "recent_status_changes_14d_bulk_adjusted",
            "suspected_bulk_issues_14d"
        ]
    ].sort_values(
        "recent_status_changes_7d",
        ascending=False
    ).head(15)
)

UPDATED WORKFLOW FEATURES


/tmp/ipykernel_4449/2863874852.py:21: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)


,instance_id,week_number,recent_status_changes_7d,recent_status_changes_7d_bulk_adjusted,recent_status_changes_14d,recent_status_changes_14d_bulk_adjusted,suspected_bulk_issues_14d
551,JSDCLOUD-I01,24,242,2,242,2,240
483,JRACLOUD-I01,22,141,7,142,8,134
497,JRASERVER-I01,3,121,0,121,0,121
465,JRACLOUD-I01,4,102,3,102,3,99
562,MC-I01,2,86,5,88,7,81
554,JSDCLOUD-I01,27,70,70,70,70,0
700,OSSRH-I01,8,56,56,103,103,0
208,ENTESB-I01,11,51,51,54,54,0
699,OSSRH-I01,7,49,49,51,51,0
579,MC-I01,19,47,47,61,61,0


## 3. Temporal event activity

Extracts rolling Jira event counts, growth, issue creation, and maturity features.

In [ ]:
# ============================================================
# NOTEBOOK 2 — STEP 3.1
# PREPARE HISTORICAL EVENT AND ISSUE CREATION RECORDS
# ============================================================

import pandas as pd
import numpy as np

KEYS = [
    "instance_id",
    "week_number",
    "checkpoint_time"
]

# Historical Jira events
historical_events = nodes_df[
    (nodes_df["label"] == "Event")
    & (nodes_df["source_type"] == "JIRA")
].copy()

historical_events = historical_events.rename(
    columns={"available_at": "event_time"}
)

historical_events["event_time"] = pd.to_datetime(
    historical_events["event_time"],
    utc=True,
    format="mixed"
)

historical_events = historical_events.drop_duplicates(
    "node_uid"
)

# Historical Jira issue creation
historical_issues = nodes_df[
    nodes_df["label"] == "HistoricalIssue"
].copy()

historical_issues = historical_issues.rename(
    columns={"available_at": "created_at"}
)

historical_issues["created_at"] = pd.to_datetime(
    historical_issues["created_at"],
    utc=True,
    format="mixed"
)

historical_issues = historical_issues.drop_duplicates(
    "node_uid"
)

print("Historical Jira events:", len(historical_events))
print("Historical Jira issues:", len(historical_issues))
print("Projects:", historical_events["instance_id"].nunique())

display(
    historical_events.groupby("instance_id")
    .size()
    .reset_index(name="event_count")
    .sort_values("event_count", ascending=False)
)

Historical Jira events: 82053
Historical Jira issues: 9000
Projects: 30


,instance_id,event_count
17,MC-I01,9078
3,CONFSERVER-I01,6639
27,SPARK-I01,6081
16,JSDCLOUD-I01,6075
15,JRASERVER-I01,5641
12,JBEAP-I01,4597
14,JRACLOUD-I01,4096
6,ENTESB-I01,3828
4,DAOS-I01,3289
7,EVG-I01,3245


In [ ]:
# ============================================================
# NOTEBOOK 2 — STEP 3.2
# ROLLING TEMPORAL EVENT FEATURES
# ============================================================

events_by_project = {
    p: g["event_time"].sort_values().to_numpy(
        dtype="datetime64[ns]"
    )
    for p, g in historical_events.groupby("instance_id")
}

records = []

for _, cp in checkpoints_df.iterrows():

    project = cp["instance_id"]
    t = cp["checkpoint_time"]

    times = events_by_project.get(
        project,
        np.array([], dtype="datetime64[ns]")
    )

    t_np = t.tz_convert("UTC").tz_localize(None).to_datetime64()

    def count_window(days):
        start = t_np - np.timedelta64(days, "D")
        return int(
            np.searchsorted(times, t_np, side="right")
            - np.searchsorted(times, start, side="right")
        )

    records.append({
        "instance_id": project,
        "week_number": cp["week_number"],
        "checkpoint_time": t,
        "events_last_7d": count_window(7),
        "events_last_14d": count_window(14),
        "events_last_28d": count_window(28)
    })

temporal_features = pd.DataFrame(records)

print("Temporal feature records:", len(temporal_features))
display(temporal_features.head(10))

Temporal feature records: 987


,instance_id,week_number,checkpoint_time,events_last_7d,events_last_14d,events_last_28d
0,AEROGEAR-I01,1,2018-01-24 05:30:01+00:00,70,71,71
1,AEROGEAR-I01,2,2018-01-31 05:30:01+00:00,30,100,101
2,AEROGEAR-I01,3,2018-02-07 05:30:01+00:00,13,43,114
3,AEROGEAR-I01,4,2018-02-14 05:30:01+00:00,40,53,153
4,AEROGEAR-I01,5,2018-02-21 05:30:01+00:00,33,73,116
5,AEROGEAR-I01,6,2018-02-28 05:30:01+00:00,27,60,113
6,AEROGEAR-I01,7,2018-03-07 05:30:01+00:00,124,151,224
7,AEROGEAR-I01,8,2018-03-14 05:30:01+00:00,126,250,310
8,AEROGEAR-I01,9,2018-03-21 05:30:01+00:00,41,167,318
9,AEROGEAR-I01,10,2018-03-28 05:30:01+00:00,9,50,300


In [ ]:
# ============================================================
# NOTEBOOK 2 — STEP 3.3
# EVENT GROWTH AND VOLATILITY
# ============================================================

extra_records = []

for _, cp in checkpoints_df.iterrows():

    project = cp["instance_id"]
    t = cp["checkpoint_time"]

    project_events = historical_events[
        historical_events["instance_id"] == project
    ]

    recent_start = t - pd.Timedelta(days=14)
    previous_start = t - pd.Timedelta(days=28)

    recent = project_events[
        (project_events["event_time"] > recent_start)
        & (project_events["event_time"] <= t)
    ]

    previous = project_events[
        (project_events["event_time"] > previous_start)
        & (project_events["event_time"] <= recent_start)
    ]

    recent_count = len(recent)
    previous_count = len(previous)

    growth = (
        (recent_count - previous_count)
        / max(previous_count, 1)
    )

    # Fixed 28-day bins relative to checkpoint
    window_events = project_events[
        (project_events["event_time"] > previous_start)
        & (project_events["event_time"] <= t)
    ]

    if len(window_events):
        elapsed_days = (
            t - window_events["event_time"]
        ).dt.total_seconds() / 86400

        day_bins = np.floor(elapsed_days).astype(int).clip(0, 27)
        daily_counts = np.bincount(
            day_bins, minlength=28
        )
    else:
        daily_counts = np.zeros(28, dtype=int)

    extra_records.append({
        "instance_id": project,
        "week_number": cp["week_number"],
        "checkpoint_time": t,
        "event_growth_ratio": growth,
        "event_activity_volatility": float(
            np.std(daily_counts)
        ),
        "active_event_days_14d": int(
            np.count_nonzero(daily_counts[:14])
        )
    })

extra_features = pd.DataFrame(extra_records)

temporal_features = temporal_features.merge(
    extra_features,
    on=KEYS,
    how="left",
    validate="one_to_one"
)

display(temporal_features.head(10))

,instance_id,week_number,checkpoint_time,events_last_7d,events_last_14d,events_last_28d,event_growth_ratio,event_activity_volatility,active_event_days_14d
0,AEROGEAR-I01,1,2018-01-24 05:30:01+00:00,70,71,71,71.000000,8.633333,5
1,AEROGEAR-I01,2,2018-01-31 05:30:01+00:00,30,100,101,99.000000,9.525677,7
2,AEROGEAR-I01,3,2018-02-07 05:30:01+00:00,13,43,114,-0.394366,9.407173,8
3,AEROGEAR-I01,4,2018-02-14 05:30:01+00:00,40,53,153,-0.470000,9.886795,11
4,AEROGEAR-I01,5,2018-02-21 05:30:01+00:00,33,73,116,0.697674,6.339864,12
5,AEROGEAR-I01,6,2018-02-28 05:30:01+00:00,27,60,113,0.132075,5.171917,12
6,AEROGEAR-I01,7,2018-03-07 05:30:01+00:00,124,151,224,1.068493,12.609520,11
7,AEROGEAR-I01,8,2018-03-14 05:30:01+00:00,126,250,310,3.166667,14.894698,10
8,AEROGEAR-I01,9,2018-03-21 05:30:01+00:00,41,167,318,0.105960,15.175672,9
9,AEROGEAR-I01,10,2018-03-28 05:30:01+00:00,9,50,300,-0.800000,15.401431,7


In [ ]:
# ============================================================
# NOTEBOOK 2 — STEP 3.4
# ISSUE CREATION AND PROJECT MATURITY
# ============================================================

creation_records = []

issues_by_project = {
    p: g["created_at"]
    for p, g in historical_issues.groupby("instance_id")
}

for _, cp in checkpoints_df.iterrows():

    project = cp["instance_id"]
    t = cp["checkpoint_time"]

    created = issues_by_project.get(
        project,
        pd.Series(dtype="datetime64[ns, UTC]")
    )

    new_14d = (
        (created > t - pd.Timedelta(days=14))
        & (created <= t)
    ).sum()

    first_created = created.min() if len(created) else pd.NaT

    project_age = (
        max((t - first_created).total_seconds() / 86400, 0)
        if pd.notna(first_created) and first_created <= t
        else 0
    )

    creation_records.append({
        "instance_id": project,
        "week_number": cp["week_number"],
        "checkpoint_time": t,
        "new_issues_last_14d": int(new_14d),
        "project_age_days": float(project_age)
    })

creation_features = pd.DataFrame(creation_records)

temporal_features = temporal_features.merge(
    creation_features,
    on=KEYS,
    how="left",
    validate="one_to_one"
)

print("FINAL TEMPORAL ACTIVITY FEATURES")

display(temporal_features.head(10))

print("\nFEATURE DISTRIBUTION")
display(
    temporal_features.drop(columns=KEYS).describe()
)

FINAL TEMPORAL ACTIVITY FEATURES


,instance_id,week_number,checkpoint_time,events_last_7d,events_last_14d,events_last_28d,event_growth_ratio,event_activity_volatility,active_event_days_14d,new_issues_last_14d,project_age_days
0,AEROGEAR-I01,1,2018-01-24 05:30:01+00:00,70,71,71,71.000000,8.633333,5,5,2274.861759
1,AEROGEAR-I01,2,2018-01-31 05:30:01+00:00,30,100,101,99.000000,9.525677,7,4,2281.861759
2,AEROGEAR-I01,3,2018-02-07 05:30:01+00:00,13,43,114,-0.394366,9.407173,8,0,2288.861759
3,AEROGEAR-I01,4,2018-02-14 05:30:01+00:00,40,53,153,-0.470000,9.886795,11,2,2295.861759
4,AEROGEAR-I01,5,2018-02-21 05:30:01+00:00,33,73,116,0.697674,6.339864,12,4,2302.861759
5,AEROGEAR-I01,6,2018-02-28 05:30:01+00:00,27,60,113,0.132075,5.171917,12,4,2309.861759
6,AEROGEAR-I01,7,2018-03-07 05:30:01+00:00,124,151,224,1.068493,12.609520,11,11,2316.861759
7,AEROGEAR-I01,8,2018-03-14 05:30:01+00:00,126,250,310,3.166667,14.894698,10,12,2323.861759
8,AEROGEAR-I01,9,2018-03-21 05:30:01+00:00,41,167,318,0.105960,15.175672,9,3,2330.861759
9,AEROGEAR-I01,10,2018-03-28 05:30:01+00:00,9,50,300,-0.800000,15.401431,7,0,2337.861759



FEATURE DISTRIBUTION


,events_last_7d,events_last_14d,events_last_28d,event_growth_ratio,event_activity_volatility,active_event_days_14d,new_issues_last_14d,project_age_days
count,987.000000,987.000000,987.000000,987.000000,987.000000,987.000000,987.000000,987.000000
mean,77.456940,153.013171,299.501520,9.921960,16.460012,9.452888,1.935157,3677.510803
std,129.587444,206.766654,344.961171,86.044282,29.044466,3.267418,2.230130,1424.008239
min,0.000000,0.000000,0.000000,-1.000000,0.000000,0.000000,0.000000,1042.718206
25%,18.000000,40.000000,86.000000,-0.345001,4.143935,7.000000,0.000000,2414.729907
50%,41.000000,87.000000,185.000000,0.026316,8.214596,10.000000,1.000000,3478.213032
75%,95.000000,192.500000,393.000000,0.745968,14.859659,12.000000,3.000000,4650.866655
max,2448.000000,2467.000000,2537.000000,2454.000000,315.170726,14.000000,18.000000,6611.953681


## 3.2 Stabilize event growth

Uses logarithmic growth to reduce extreme values from low baseline activity.

In [ ]:
# ============================================================
# NOTEBOOK 2 — STEP 3.6
# STABILIZE TEMPORAL ACTIVITY FEATURES
# ============================================================

import numpy as np

# Previous 14-day event count
temporal_features["events_previous_14d"] = (
    temporal_features["events_last_28d"]
    - temporal_features["events_last_14d"]
)

# Logarithmic activity growth
temporal_features["event_log_growth"] = (
    np.log1p(temporal_features["events_last_14d"])
    -
    np.log1p(temporal_features["events_previous_14d"])
)

# Log-transformed activity volume
for col in [
    "events_last_7d",
    "events_last_14d",
    "events_last_28d"
]:
    temporal_features[f"log_{col}"] = np.log1p(
        temporal_features[col]
    )

print("STABILIZED TEMPORAL FEATURES")

display(
    temporal_features[
        [
            "instance_id",
            "week_number",
            "events_last_14d",
            "events_previous_14d",
            "event_growth_ratio",
            "event_log_growth"
        ]
    ].head(15)
)

print("\nLOG GROWTH DISTRIBUTION")

display(
    temporal_features[
        ["event_growth_ratio", "event_log_growth"]
    ].describe()
)

STABILIZED TEMPORAL FEATURES


,instance_id,week_number,events_last_14d,events_previous_14d,event_growth_ratio,event_log_growth
0,AEROGEAR-I01,1,71,0,71.000000,4.276666
1,AEROGEAR-I01,2,100,1,99.000000,3.921973
2,AEROGEAR-I01,3,43,71,-0.394366,-0.492476
3,AEROGEAR-I01,4,53,100,-0.470000,-0.626136
4,AEROGEAR-I01,5,73,43,0.697674,0.519875
5,AEROGEAR-I01,6,60,53,0.132075,0.121890
6,AEROGEAR-I01,7,151,73,1.068493,0.719815
7,AEROGEAR-I01,8,250,60,3.166667,1.414579
8,AEROGEAR-I01,9,167,151,0.105960,0.100083
9,AEROGEAR-I01,10,50,250,-0.800000,-1.593627



LOG GROWTH DISTRIBUTION


,event_growth_ratio,event_log_growth
count,987.000000,987.000000
mean,9.921960,0.237173
std,86.044282,1.370796
min,-1.000000,-3.931826
25%,-0.345001,-0.416201
50%,0.026316,0.025891
75%,0.745968,0.545635
max,2454.000000,7.805882


## 3.3 Candidate bulk event adjustments

Screens event-minute concentrations and creates adjusted activity features.

In [ ]:
# ============================================================
# NOTEBOOK 2 — STEP 3.9
# IDENTIFY SUSPICIOUS EVENT BATCHES
# ============================================================

import pandas as pd
import numpy as np

MIN_EVENTS_PER_MINUTE = 50
MIN_ISSUES_PER_MINUTE = 10

event_audit = historical_events.copy()

event_audit["event_minute"] = (
    event_audit["event_time"].dt.floor("min")
)

minute_activity = (
    event_audit.groupby(
        ["instance_id", "event_minute"]
    )
    .agg(
        event_count=("node_uid", "nunique"),
        affected_issues=("source_id", "nunique")
    )
    .reset_index()
)

minute_activity["suspected_bulk"] = (
    (minute_activity["event_count"] >= MIN_EVENTS_PER_MINUTE)
    &
    (minute_activity["affected_issues"] >= MIN_ISSUES_PER_MINUTE)
)

suspected_minutes = minute_activity[
    minute_activity["suspected_bulk"]
].copy()

print("Candidate bulk minutes:", len(suspected_minutes))
print("Projects affected:",
      suspected_minutes["instance_id"].nunique())

display(
    suspected_minutes.sort_values(
        "event_count",
        ascending=False
    ).head(30)
)

Candidate bulk minutes: 50
Projects affected: 14


,instance_id,event_minute,event_count,affected_issues,suspected_bulk
15520,JBIDE-I01,2020-09-14 05:28:00+00:00,666,98,True
15521,JBIDE-I01,2020-09-14 05:29:00+00:00,432,67,True
8605,ENTESB-I01,2020-09-14 05:38:00+00:00,430,75,True
29970,RUBY-I01,2021-04-16 03:58:00+00:00,270,270,True
21151,JSDCLOUD-I01,2019-09-19 06:09:00+00:00,249,180,True
21017,JSDCLOUD-I01,2019-08-26 03:06:00+00:00,174,89,True
27353,MONGOID-I01,2021-04-16 03:56:00+00:00,171,171,True
8595,ENTESB-I01,2020-09-13 10:42:00+00:00,168,22,True
13318,JBDS-I01,2016-09-14 11:15:00+00:00,138,114,True
29720,QTCREATORBUG-I01,2018-08-20 13:03:00+00:00,131,131,True


In [ ]:
# ============================================================
# NOTEBOOK 2 — STEP 3.10
# BULK-ADJUSTED TEMPORAL ACTIVITY FEATURES
# ============================================================

# Mark events occurring in suspicious minutes
bulk_keys = suspected_minutes[
    ["instance_id", "event_minute"]
].drop_duplicates()

bulk_keys["suspected_bulk"] = True

adjusted_events = event_audit.merge(
    bulk_keys,
    on=["instance_id", "event_minute"],
    how="left",
    validate="many_to_one"
)

adjusted_events["suspected_bulk"] = (
    adjusted_events["suspected_bulk"]
    .eq(True)
)

clean_events = adjusted_events[
    ~adjusted_events["suspected_bulk"]
].copy()

print("Original events:", len(event_audit))
print("Flagged events:",
      int(adjusted_events["suspected_bulk"].sum()))
print("Remaining events:", len(clean_events))

# Build project-level timestamp arrays
clean_times = {
    p: np.sort(
        g["event_time"]
        .dt.tz_convert("UTC")
        .dt.tz_localize(None)
        .to_numpy(dtype="datetime64[ns]")
    )
    for p, g in clean_events.groupby("instance_id")
}

records = []

for cp in checkpoints_df.itertuples(index=False):

    project = cp.instance_id
    t = cp.checkpoint_time

    times = clean_times.get(
        project,
        np.array([], dtype="datetime64[ns]")
    )

    t_np = t.tz_convert("UTC").tz_localize(None).to_datetime64()

    def count_window(days):
        start = t_np - np.timedelta64(days, "D")
        return int(
            np.searchsorted(times, t_np, side="right")
            - np.searchsorted(times, start, side="right")
        )

    count_7 = count_window(7)
    count_14 = count_window(14)
    count_28 = count_window(28)

    previous_14 = count_28 - count_14

    # Daily event counts in 28 consecutive 24-hour bins
    start_28 = t_np - np.timedelta64(28, "D")

    left = np.searchsorted(
        times, start_28, side="right"
    )
    right = np.searchsorted(
        times, t_np, side="right"
    )

    window_times = times[left:right]

    if len(window_times):
        elapsed = (
            (t_np - window_times)
            / np.timedelta64(1, "D")
        )

        bins = np.floor(elapsed).astype(int)
        bins = np.clip(bins, 0, 27)

        daily_counts = np.bincount(
            bins, minlength=28
        )
    else:
        daily_counts = np.zeros(28, dtype=int)

    records.append({
        "instance_id": project,
        "week_number": cp.week_number,
        "checkpoint_time": t,
        "events_last_7d_bulk_adjusted": count_7,
        "events_last_14d_bulk_adjusted": count_14,
        "events_last_28d_bulk_adjusted": count_28,
        "event_log_growth_bulk_adjusted": (
            np.log1p(count_14)
            - np.log1p(previous_14)
        ),
        "event_volatility_bulk_adjusted": float(
            np.std(daily_counts)
        ),
        "active_event_days_14d_bulk_adjusted": int(
            np.count_nonzero(daily_counts[:14])
        )
    })

bulk_temporal_features = pd.DataFrame(records)

temporal_features = temporal_features.merge(
    bulk_temporal_features,
    on=KEYS,
    how="left",
    validate="one_to_one"
)

print("Updated temporal records:", len(temporal_features))

display(
    temporal_features[
        [
            "instance_id",
            "week_number",
            "events_last_7d",
            "events_last_7d_bulk_adjusted",
            "events_last_14d",
            "events_last_14d_bulk_adjusted",
            "event_log_growth",
            "event_log_growth_bulk_adjusted"
        ]
    ].sort_values(
        "events_last_7d",
        ascending=False
    ).head(15)
)

Original events: 82053
Flagged events: 5920
Remaining events: 76133
Updated temporal records: 987


,instance_id,week_number,events_last_7d,events_last_7d_bulk_adjusted,events_last_14d,events_last_14d_bulk_adjusted,event_log_growth,event_log_growth_bulk_adjusted
440,JBIDE-I01,12,2448,221,2454,227,7.805882,5.429346
220,ENTESB-I01,23,1103,310,1115,322,3.028522,1.788668
571,MC-I01,11,986,986,1399,1399,1.035637,1.035637
564,MC-I01,4,895,895,1269,1269,0.962623,0.962623
895,SPARK-I01,4,795,795,830,830,3.390425,3.390425
105,CONFSERVER-I01,7,715,715,1254,1254,0.402680,0.402680
101,CONFSERVER-I01,3,715,715,737,737,1.027995,1.247358
551,JSDCLOUD-I01,24,698,410,831,543,0.996985,0.572101
24,AEROGEAR-I01,25,584,256,611,283,2.982745,2.214987
104,CONFSERVER-I01,6,539,539,925,925,-0.232174,-0.232174


## 4. Dependency graph features

Builds directed dependency graphs from checkpoint-active blocking links.

In [ ]:
# ============================================================
# NOTEBOOK 2 — STEP 4.1
# PREPARE HISTORICAL DEPENDENCY GRAPH RECORDS
# ============================================================

import pandas as pd
import numpy as np
import networkx as nx

KEYS = [
    "instance_id",
    "week_number",
    "checkpoint_time"
]

dependency_graph_data = dependencies_df.copy()

dependency_graph_data = dependency_graph_data.drop_duplicates(
    KEYS + ["upstream_issue", "downstream_issue"]
)

dependency_groups = {
    (project, week): group
    for (project, week), group in dependency_graph_data.groupby(
        ["instance_id", "week_number"]
    )
}

print("Dependency records:", len(dependency_graph_data))
print("Project-weeks with dependencies:", len(dependency_groups))
print("Projects:", dependency_graph_data["instance_id"].nunique())

display(dependency_graph_data.head())

Dependency records: 12684
Project-weeks with dependencies: 441
Projects: 16


,instance_id,week_number,checkpoint_time,upstream_issue,downstream_issue,active_from,start_event_uid
0,AEROGEAR-I01,5,2018-02-21 05:30:01+00:00,AEROGEAR-4347,AEROGEAR-1968,2018-02-15 05:23:04+00:00,T_7d51b32c251b512fb86d1559
1,AEROGEAR-I01,5,2018-02-21 05:30:01+00:00,AGDROID-683,AEROGEAR-1968,2018-02-15 05:23:04+00:00,T_2f188ec3f5fc9f698213b945
2,AEROGEAR-I01,6,2018-02-28 05:30:01+00:00,AEROGEAR-4347,AEROGEAR-1968,2018-02-15 05:23:04+00:00,T_7d51b32c251b512fb86d1559
3,AEROGEAR-I01,6,2018-02-28 05:30:01+00:00,AGDROID-683,AEROGEAR-1968,2018-02-15 05:23:04+00:00,T_2f188ec3f5fc9f698213b945
4,AEROGEAR-I01,7,2018-03-07 05:30:01+00:00,AEROGEAR-2210,AEROGEAR-2218,2018-03-01 09:46:57+00:00,T_62d9e53fc092cf712ba348bf


In [ ]:
# ============================================================
# NOTEBOOK 2 — STEP 4.2
# EXTRACT DIRECTED DEPENDENCY GRAPH FEATURES
# ============================================================

def calculate_dependency_features(group, total_issues):

    G = nx.DiGraph()

    if group is not None and len(group) > 0:
        G.add_edges_from(
            group[
                ["upstream_issue", "downstream_issue"]
            ].itertuples(index=False, name=None)
        )

    edge_count = G.number_of_edges()
    node_count = G.number_of_nodes()

    if edge_count == 0:
        return {
            "dependency_links": 0,
            "blocked_issue_count": 0,
            "blocking_issue_count": 0,
            "dependency_density": 0.0,
            "max_blocking_degree": 0,
            "avg_blocking_degree": 0.0,
            "max_blocker_count": 0,
            "dependency_components": 0,
            "largest_dependency_component": 0,
            "max_dependency_chain": 0,
            "dependency_cycle_count": 0
        }

    out_degrees = dict(G.out_degree())
    in_degrees = dict(G.in_degree())

    blocking_nodes = [
        n for n, degree in out_degrees.items()
        if degree > 0
    ]

    blocked_nodes = [
        n for n, degree in in_degrees.items()
        if degree > 0
    ]

    # Weakly connected components, excluding isolated issues
    weak_components = list(
        nx.weakly_connected_components(G)
    )

    # Strongly connected components identify cycles
    scc = list(nx.strongly_connected_components(G))

    cyclic_components = sum(
        1 for component in scc
        if len(component) > 1
        or any(G.has_edge(n, n) for n in component)
    )

    # Condensation DAG supports cyclic dependency graphs
    condensed = nx.condensation(G)

    max_chain = (
        nx.dag_longest_path_length(condensed)
        if condensed.number_of_edges() > 0
        else 0
    )

    # Density over all historical Jira issues
    # Avoid using only linked nodes as denominator
    density = (
        edge_count / (total_issues * (total_issues - 1))
        if total_issues > 1
        else 0.0
    )

    return {
        "dependency_links": edge_count,
        "blocked_issue_count": len(blocked_nodes),
        "blocking_issue_count": len(blocking_nodes),
        "dependency_density": density,
        "max_blocking_degree": max(out_degrees.values()),
        "avg_blocking_degree": (
            edge_count / len(blocking_nodes)
        ),
        "max_blocker_count": max(in_degrees.values()),
        "dependency_components": len(weak_components),
        "largest_dependency_component": max(
            len(c) for c in weak_components
        ),
        "max_dependency_chain": max_chain,
        "dependency_cycle_count": cyclic_components
    }


records = []

for cp in checkpoints_df.itertuples(index=False):

    key = (cp.instance_id, cp.week_number)

    group = dependency_groups.get(key)

    features = calculate_dependency_features(
        group,
        cp.historical_issues
    )

    records.append({
        "instance_id": cp.instance_id,
        "week_number": cp.week_number,
        "checkpoint_time": cp.checkpoint_time,
        **features
    })

dependency_features = pd.DataFrame(records)

print("Dependency feature records:", len(dependency_features))

display(dependency_features.head(10))

print("\nFEATURE DISTRIBUTION")
display(
    dependency_features.drop(columns=KEYS).describe()
)

Dependency feature records: 987


,instance_id,week_number,checkpoint_time,dependency_links,blocked_issue_count,blocking_issue_count,dependency_density,max_blocking_degree,avg_blocking_degree,max_blocker_count,dependency_components,largest_dependency_component,max_dependency_chain,dependency_cycle_count
0,AEROGEAR-I01,1,2018-01-24 05:30:01+00:00,0,0,0,0.000000,0,0.000000,0,0,0,0,0
1,AEROGEAR-I01,2,2018-01-31 05:30:01+00:00,0,0,0,0.000000,0,0.000000,0,0,0,0,0
2,AEROGEAR-I01,3,2018-02-07 05:30:01+00:00,0,0,0,0.000000,0,0.000000,0,0,0,0,0
3,AEROGEAR-I01,4,2018-02-14 05:30:01+00:00,0,0,0,0.000000,0,0.000000,0,0,0,0,0
4,AEROGEAR-I01,5,2018-02-21 05:30:01+00:00,2,1,2,0.000073,1,1.000000,2,1,3,1,0
5,AEROGEAR-I01,6,2018-02-28 05:30:01+00:00,2,1,2,0.000071,1,1.000000,2,1,3,1,0
6,AEROGEAR-I01,7,2018-03-07 05:30:01+00:00,17,8,14,0.000546,3,1.214286,4,5,5,2,0
7,AEROGEAR-I01,8,2018-03-14 05:30:01+00:00,20,10,17,0.000621,3,1.176471,4,6,5,3,0
8,AEROGEAR-I01,9,2018-03-21 05:30:01+00:00,20,10,17,0.000621,3,1.176471,4,6,5,3,0
9,AEROGEAR-I01,10,2018-03-28 05:30:01+00:00,20,10,17,0.000621,3,1.176471,4,6,5,3,0



FEATURE DISTRIBUTION


,dependency_links,blocked_issue_count,blocking_issue_count,dependency_density,max_blocking_degree,avg_blocking_degree,max_blocker_count,dependency_components,largest_dependency_component,max_dependency_chain,dependency_cycle_count
count,987.000000,987.000000,987.000000,987.000000,987.000000,987.000000,987.000000,987.000000,987.000000,987.000000,987.0
mean,12.851064,6.579534,9.579534,0.000680,1.591692,0.547412,2.356636,3.044580,4.346505,0.710233,0.0
std,45.544988,19.250366,32.119113,0.003196,4.357555,0.660631,7.925008,6.073422,15.121197,0.923618,0.0
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
50%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
75%,6.000000,5.000000,5.000000,0.000105,2.000000,1.093750,1.000000,3.000000,3.000000,1.000000,0.0
max,353.000000,135.000000,256.000000,0.021889,27.000000,5.000000,56.000000,34.000000,106.000000,3.000000,0.0


## 4.2 Dependency issue coverage

Identifies references missing from the sampled historical Jira issue registry.

In [ ]:
# ============================================================
# NOTEBOOK 2 — STEP 4.4
# AUDIT DEPENDENCY ISSUE COVERAGE
# ============================================================

import pandas as pd
import numpy as np

KEYS = ["instance_id", "week_number", "checkpoint_time"]

# Historical issue identities from Notebook 1
known_issues = nodes_df[
    nodes_df["label"] == "HistoricalIssue"
][["instance_id", "source_id", "available_at"]].copy()

known_issues = known_issues.rename(columns={
    "source_id": "issue_key",
    "available_at": "created_at"
})

known_issues["created_at"] = pd.to_datetime(
    known_issues["created_at"], utc=True
)

# Check each endpoint against the local project issue registry
known_lookup = known_issues.set_index(
    ["instance_id", "issue_key"]
)["created_at"]

audit = dependencies_df.copy()

for role, col in [
    ("upstream", "upstream_issue"),
    ("downstream", "downstream_issue")
]:
    keys = pd.MultiIndex.from_frame(
        audit[["instance_id", col]].rename(
            columns={col: "issue_key"}
        )
    )

    audit[f"{role}_created_at"] = known_lookup.reindex(
        keys
    ).to_numpy()

    audit[f"{role}_known_at_checkpoint"] = (
        audit[f"{role}_created_at"].notna()
        &
        (
            audit[f"{role}_created_at"]
            <= audit["checkpoint_time"]
        )
    )

print("DEPENDENCY ENDPOINT COVERAGE")

display(pd.DataFrame({
    "endpoint": ["Upstream", "Downstream"],
    "known_records": [
        audit["upstream_known_at_checkpoint"].sum(),
        audit["downstream_known_at_checkpoint"].sum()
    ],
    "unmatched_records": [
        (~audit["upstream_known_at_checkpoint"]).sum(),
        (~audit["downstream_known_at_checkpoint"]).sum()
    ]
}))

# Identify checkpoints exceeding sampled issue counts
comparison = dependency_features.merge(
    checkpoints_df[
        KEYS + ["historical_issues"]
    ],
    on=KEYS,
    validate="one_to_one"
)

invalid = comparison[
    (comparison["blocked_issue_count"] >
     comparison["historical_issues"])
    |
    (comparison["blocking_issue_count"] >
     comparison["historical_issues"])
]

print("\nCHECKPOINTS EXCEEDING SAMPLED ISSUE COUNT:",
      len(invalid))

display(
    invalid[
        [
            "instance_id",
            "week_number",
            "historical_issues",
            "blocked_issue_count",
            "blocking_issue_count"
        ]
    ].head(20)
)

DEPENDENCY ENDPOINT COVERAGE


,endpoint,known_records,unmatched_records
0,Upstream,5324,7360
1,Downstream,9668,3016



CHECKPOINTS EXCEEDING SAMPLED ISSUE COUNT: 22


,instance_id,week_number,historical_issues,blocked_issue_count,blocking_issue_count
407,JBEAP-I01,12,102,68,122
408,JBEAP-I01,13,105,88,127
409,JBEAP-I01,14,108,91,131
410,JBEAP-I01,15,109,99,139
411,JBEAP-I01,16,110,112,139
412,JBEAP-I01,17,112,114,150
413,JBEAP-I01,18,112,115,153
414,JBEAP-I01,19,114,117,181
415,JBEAP-I01,20,116,124,195
416,JBEAP-I01,21,116,125,197


In [ ]:
# ============================================================
# NOTEBOOK 2 — STEP 4.5
# COVERAGE-AWARE DEPENDENCY FEATURES
# ============================================================

coverage_records = []

for cp in checkpoints_df.itertuples(index=False):

    group = audit[
        (audit["instance_id"] == cp.instance_id)
        &
        (audit["week_number"] == cp.week_number)
    ]

    upstream_known = set(
        group.loc[
            group["upstream_known_at_checkpoint"],
            "upstream_issue"
        ]
    )

    downstream_known = set(
        group.loc[
            group["downstream_known_at_checkpoint"],
            "downstream_issue"
        ]
    )

    all_nodes = set(group["upstream_issue"]) | set(
        group["downstream_issue"]
    )

    known_nodes = upstream_known | downstream_known

    coverage_records.append({
        "instance_id": cp.instance_id,
        "week_number": cp.week_number,
        "checkpoint_time": cp.checkpoint_time,
        "dependency_graph_nodes": len(all_nodes),
        "known_dependency_nodes": len(known_nodes),
        "unmatched_dependency_nodes": len(
            all_nodes - known_nodes
        ),
        "dependency_node_coverage": (
            len(known_nodes) / len(all_nodes)
            if all_nodes else 0.0
        )
    })

coverage_features = pd.DataFrame(coverage_records)

dependency_features = dependency_features.merge(
    coverage_features,
    on=KEYS,
    how="left",
    validate="one_to_one"
)

# Density among nodes actually represented in the graph
n = dependency_features["dependency_graph_nodes"]

dependency_features["dependency_graph_density"] = np.where(
    n > 1,
    dependency_features["dependency_links"]
    / (n * (n - 1)).clip(lower=1),
    0.0
)

display(
    dependency_features[
        [
            "instance_id",
            "week_number",
            "dependency_links",
            "dependency_graph_nodes",
            "known_dependency_nodes",
            "unmatched_dependency_nodes",
            "dependency_node_coverage",
            "dependency_graph_density"
        ]
    ].head(15)
)

,instance_id,week_number,dependency_links,dependency_graph_nodes,known_dependency_nodes,unmatched_dependency_nodes,dependency_node_coverage,dependency_graph_density
0,AEROGEAR-I01,1,0,0,0,0,0.000000,0.000000
1,AEROGEAR-I01,2,0,0,0,0,0.000000,0.000000
2,AEROGEAR-I01,3,0,0,0,0,0.000000,0.000000
3,AEROGEAR-I01,4,0,0,0,0,0.000000,0.000000
4,AEROGEAR-I01,5,2,3,2,1,0.666667,0.333333
5,AEROGEAR-I01,6,2,3,2,1,0.666667,0.333333
6,AEROGEAR-I01,7,17,19,12,7,0.631579,0.049708
7,AEROGEAR-I01,8,20,23,13,10,0.565217,0.039526
8,AEROGEAR-I01,9,20,23,13,10,0.565217,0.039526
9,AEROGEAR-I01,10,20,23,13,10,0.565217,0.039526


## 5. Temporal knowledge graph structure

Calculates event–entity connectivity and growth features.

In [ ]:
# ============================================================
# NOTEBOOK 2 — STEP 5.1
# PREPARE TEMPORAL EVENT-ENTITY GRAPH
# ============================================================

import pandas as pd
import numpy as np

KEYS = [
    "instance_id",
    "week_number",
    "checkpoint_time"
]

# Use historical event-entity references
kg_references = edges_df[
    edges_df["type"] == "REFERS_TO"
].copy()

# Retain relevant node types
kg_entities = nodes_df[
    nodes_df["label"] == "CanonicalEntity"
].copy()

kg_events = nodes_df[
    nodes_df["label"] == "Event"
].copy()

print("Event-entity relationships:", len(kg_references))
print("Canonical entities:", len(kg_entities))
print("Historical events:", len(kg_events))

print("\nRelationship project coverage:")
display(
    kg_references.groupby("instance_id")
    .size()
    .reset_index(name="relationships")
    .sort_values("relationships", ascending=False)
)

Event-entity relationships: 82053
Canonical entities: 5434
Historical events: 82053

Relationship project coverage:


,instance_id,relationships
17,MC-I01,9078
3,CONFSERVER-I01,6639
27,SPARK-I01,6081
16,JSDCLOUD-I01,6075
15,JRASERVER-I01,5641
12,JBEAP-I01,4597
14,JRACLOUD-I01,4096
6,ENTESB-I01,3828
4,DAOS-I01,3289
7,EVG-I01,3245


In [ ]:
# ============================================================
# NOTEBOOK 2 — STEP 5.2
# EXTRACT TEMPORAL KG STRUCTURAL FEATURES
# ============================================================

nodes_by_project = {
    p: g.copy()
    for p, g in nodes_df.groupby("instance_id")
}

refs_by_project = {
    p: g.copy()
    for p, g in kg_references.groupby("instance_id")
}

records = []

for cp in checkpoints_df.itertuples(index=False):

    project = cp.instance_id
    t = cp.checkpoint_time

    project_nodes = nodes_by_project[project]

    visible_nodes = project_nodes[
        project_nodes["available_at"] <= t
    ]

    project_refs = refs_by_project.get(
        project,
        kg_references.iloc[0:0]
    )

    visible_refs = project_refs[
        project_refs["available_at"] <= t
    ]

    entity_nodes = visible_nodes[
        visible_nodes["label"] == "CanonicalEntity"
    ]

    event_nodes = visible_nodes[
        visible_nodes["label"] == "Event"
    ]

    # Entity reference degree
    entity_degrees = (
        visible_refs["to_uid"].value_counts()
    )

    max_degree = (
        int(entity_degrees.max())
        if len(entity_degrees) else 0
    )

    avg_degree = (
        float(entity_degrees.mean())
        if len(entity_degrees) else 0.0
    )

    concentration = (
        max_degree / len(visible_refs)
        if len(visible_refs) else 0.0
    )

    start_14d = t - pd.Timedelta(days=14)

    recent_refs = visible_refs[
        visible_refs["available_at"] > start_14d
    ]

    new_entities = entity_nodes[
        entity_nodes["available_at"] > start_14d
    ]

    records.append({
        "instance_id": project,
        "week_number": cp.week_number,
        "checkpoint_time": t,

        "kg_visible_nodes": len(visible_nodes),
        "kg_visible_edges": int(cp.kg_edges),
        "kg_event_nodes": len(event_nodes),
        "kg_entity_nodes": len(entity_nodes),

        "kg_active_entities_14d": (
            recent_refs["to_uid"].nunique()
        ),

        "kg_max_entity_degree": max_degree,
        "kg_avg_entity_degree": avg_degree,
        "kg_entity_concentration": concentration,

        "kg_new_entities_14d": len(new_entities),
        "kg_edge_growth_14d": len(recent_refs)
    })

kg_features = pd.DataFrame(records)

print("Structural KG feature records:", len(kg_features))

display(kg_features.head(10))

print("\nFEATURE DISTRIBUTION")
display(
    kg_features.drop(columns=KEYS).describe()
)

Structural KG feature records: 987


,instance_id,week_number,checkpoint_time,kg_visible_nodes,kg_visible_edges,kg_event_nodes,kg_entity_nodes,kg_active_entities_14d,kg_max_entity_degree,kg_avg_entity_degree,kg_entity_concentration,kg_new_entities_14d,kg_edge_growth_14d
0,AEROGEAR-I01,1,2018-01-24 05:30:01+00:00,239,142,71,6,6,42,11.833333,0.591549,6,71
1,AEROGEAR-I01,2,2018-01-31 05:30:01+00:00,271,202,101,8,8,42,12.625000,0.415842,7,100
2,AEROGEAR-I01,3,2018-02-07 05:30:01+00:00,284,228,114,8,7,42,14.250000,0.368421,2,43
3,AEROGEAR-I01,4,2018-02-14 05:30:01+00:00,328,308,154,10,6,42,15.400000,0.272727,2,53
4,AEROGEAR-I01,5,2018-02-21 05:30:01+00:00,365,374,187,12,11,42,15.583333,0.224599,4,73
5,AEROGEAR-I01,6,2018-02-28 05:30:01+00:00,396,428,214,14,10,42,15.285714,0.196262,4,60
6,AEROGEAR-I01,7,2018-03-07 05:30:01+00:00,538,676,338,23,16,43,14.695652,0.127219,11,151
7,AEROGEAR-I01,8,2018-03-14 05:30:01+00:00,670,928,464,26,21,46,17.846154,0.099138,12,250
8,AEROGEAR-I01,9,2018-03-21 05:30:01+00:00,711,1010,505,26,18,46,19.423077,0.091089,3,167
9,AEROGEAR-I01,10,2018-03-28 05:30:01+00:00,720,1028,514,26,6,46,19.769231,0.089494,0,50



FEATURE DISTRIBUTION


,kg_visible_nodes,kg_visible_edges,kg_event_nodes,kg_entity_nodes,kg_active_entities_14d,kg_max_entity_degree,kg_avg_entity_degree,kg_entity_concentration,kg_new_entities_14d,kg_edge_growth_14d
count,987.000000,987.000000,987.000000,987.000000,987.000000,987.000000,987.000000,987.000000,987.000000,987.000000
mean,1790.075988,2885.753799,1442.878419,115.936170,31.497467,100.768997,13.701250,0.104147,10.273556,153.013171
std,1612.221813,3115.384624,1557.693770,85.272888,43.833341,139.279548,12.009370,0.121250,31.112440,206.766654
min,88.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,693.500000,807.000000,403.500000,45.000000,8.000000,36.000000,5.851332,0.043956,1.000000,40.000000
50%,1230.000000,1752.000000,876.000000,90.000000,15.000000,57.000000,10.357143,0.067376,3.000000,87.000000
75%,2312.500000,3914.000000,1957.000000,173.000000,38.000000,97.500000,17.169827,0.117897,6.000000,192.500000
max,9185.000000,17472.000000,8736.000000,295.000000,279.000000,825.000000,76.126984,1.000000,277.000000,2467.000000


## 6. Historical scope-change features

Loads the 938 final structural scope events and derives past-only scope indicators.

In [ ]:
# ============================================================
# NOTEBOOK 2 — STEP 6.2
# LOAD AND VALIDATE HISTORICAL SCOPE EVENTS
# ============================================================

import pandas as pd
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/GC1")

SCOPE_FILE = (
    ROOT / "Temporal_snapshots"
    / "historical_structural_scope_events.parquet"
)

scope_events_df = pd.read_parquet(SCOPE_FILE)

# Normalize timestamps
scope_events_df["event_dt"] = pd.to_datetime(
    scope_events_df["event_dt"],
    utc=True,
    errors="coerce",
    format="mixed"
)

print("=" * 60)
print("HISTORICAL SCOPE EVENT DATA")
print("=" * 60)

print("Total events:", len(scope_events_df))
print("Projects:", scope_events_df["instance_id"].nunique())

print("\nAvailable columns:")
print(scope_events_df.columns.tolist())

print("\nStructural event distribution:")
display(scope_events_df["structural_class"].value_counts())

print("\nSample records:")
display(scope_events_df.head(10))

# Validations
assert len(scope_events_df) == 938, (
    f"Expected 938 events, found {len(scope_events_df)}"
)

assert scope_events_df["instance_id"].nunique() == 26

assert scope_events_df["event_dt"].notna().all()

assert scope_events_df["uid"].is_unique

print("\nALL SCOPE EVENT INPUT VALIDATIONS PASSED")

HISTORICAL SCOPE EVENT DATA
Total events: 938
Projects: 26

Available columns:
['canonical_id', 'entity_type', 'instance_id', 'label', 'name', 'payload_json', 'project_key', 'uid', 'created_at', 'observed_at', 'temporal_basis', 'temporal_status', 'chunk_id', 'source_id', 'source_timestamp', 'source_type', 'evidence_text', 'event_time', 'valid_from', 'valid_to', 'relation_type', 'event_type', 'jira_field', 'from_value', 'to_value', 'has_observed_at', 'event_dt', 'old_normalized', 'new_normalized', 'has_actual_change', 'has_both_values', 'structural_class', 'affected_issue', 'event_week', 'suspected_admin_reclassification']

Structural event distribution:


,count
structural_class,
EPIC_CHILD_ADDED,374
EPIC_ASSIGNED,247
ISSUE_RECLASSIFIED,161
EPIC_REASSIGNED,71
EPIC_UNASSIGNED,49
EPIC_CHILD_REMOVED,36



Sample records:


,canonical_id,entity_type,instance_id,label,name,payload_json,project_key,uid,created_at,observed_at,...,has_observed_at,event_dt,old_normalized,new_normalized,has_actual_change,has_both_values,structural_class,affected_issue,event_week,suspected_admin_reclassification
0,None,None,FLEX-I01,Event,None,"{""actor"": ""<<|author_displayName|6176ef55-35bb...",None,T_a0ea0d42c41035f928a8a784,None,2013-09-16T13:32:53+00:00,...,True,2013-09-16 13:32:53+00:00,bug,improvement,True,True,ISSUE_RECLASSIFIED,FLEX-33732,2013-09-16,False
1,None,None,FLEX-I01,Event,None,"{""actor"": ""<<|author_displayName|51d8b8b4-d2a7...",None,T_aad9f88b4a15180ff71f5818,None,2013-09-23T09:37:40+00:00,...,True,2013-09-23 09:37:40+00:00,task,sub-task,True,True,ISSUE_RECLASSIFIED,FLEX-33299,2013-09-23,False
2,None,None,FLEX-I01,Event,None,"{""actor"": ""<<|author_displayName|2af44677-47c0...",None,T_1d8e0e40fd2726fd18ce78c6,None,2013-11-18T00:04:00+00:00,...,True,2013-11-18 00:04:00+00:00,improvement,bug,True,True,ISSUE_RECLASSIFIED,FLEX-33166,2013-11-18,False
3,None,None,FLEX-I01,Event,None,"{""actor"": ""<<|author_displayName|23195960-99e9...",None,T_b33c47b219b05fa2d4d6dfc6,None,2014-01-16T12:09:13+00:00,...,True,2014-01-16 12:09:13+00:00,bug,sub-task,True,True,ISSUE_RECLASSIFIED,FLEX-26768,2014-01-13,False
4,None,None,FLEX-I01,Event,None,"{""actor"": ""<<|author_displayName|6176ef55-35bb...",None,T_e088a8cecf6b10302354d470,None,2014-05-04T02:35:57+00:00,...,True,2014-05-04 02:35:57+00:00,bug,improvement,True,True,ISSUE_RECLASSIFIED,FLEX-33539,2014-04-28,False
5,None,None,FLEX-I01,Event,None,"{""actor"": ""<<|author_displayName|e3e2ba1f-ca33...",None,T_ba11ee6443b864549024efbd,None,2014-05-05T19:59:57+00:00,...,True,2014-05-05 19:59:57+00:00,new feature,improvement,True,True,ISSUE_RECLASSIFIED,FLEX-34308,2014-05-05,False
6,None,None,JBEAP-I01,Event,None,"{""actor"": ""<<|author_displayName|43c5a258-6e7f...",None,T_45cecf1cbe02fe68833e9666,None,2015-12-01T12:27:34+00:00,...,True,2015-12-01 12:27:34+00:00,documentation,bug,True,True,ISSUE_RECLASSIFIED,JBEAP-1681,2015-11-30,False
7,None,None,JBEAP-I01,Event,None,"{""actor"": ""<<|author_displayName|43c5a258-6e7f...",None,T_a4aee335d6647b74d0189a58,None,2015-12-01T12:44:05+00:00,...,True,2015-12-01 12:44:05+00:00,documentation,bug,True,True,ISSUE_RECLASSIFIED,JBEAP-1848,2015-11-30,False
8,None,None,JBEAP-I01,Event,None,"{""actor"": ""<<|author_displayName|43c5a258-6e7f...",None,T_be3ae81a704c6434292ee3b8,None,2015-12-01T13:31:56+00:00,...,True,2015-12-01 13:31:56+00:00,documentation,bug,True,True,ISSUE_RECLASSIFIED,JBEAP-105,2015-11-30,False
9,None,None,JBEAP-I01,Event,None,"{""actor"": ""<<|author_displayName|43c5a258-6e7f...",None,T_8d8de5b39bb550841818fb0c,None,2015-12-01T14:00:58+00:00,...,True,2015-12-01 14:00:58+00:00,documentation,bug,True,True,ISSUE_RECLASSIFIED,JBEAP-1483,2015-11-30,False



ALL SCOPE EVENT INPUT VALIDATIONS PASSED


In [ ]:
# ============================================================
# NOTEBOOK 2 — STEP 6.3
# HISTORICAL SCOPE-CHANGE FEATURE ENGINEERING
# ============================================================

import pandas as pd
import numpy as np

KEYS = [
    "instance_id",
    "week_number",
    "checkpoint_time"
]

scope_events = scope_events_df.copy()

scope_events["event_dt"] = pd.to_datetime(
    scope_events["event_dt"],
    utc=True,
    errors="coerce",
    format="mixed"
)

scope_groups = {
    project: group.sort_values("event_dt")
    for project, group in scope_events.groupby("instance_id")
}

records = []

for cp in checkpoints_df.itertuples(index=False):

    project = cp.instance_id
    t = cp.checkpoint_time

    project_events = scope_groups.get(
        project,
        scope_events.iloc[0:0]
    )

    def get_window(days):
        start = t - pd.Timedelta(days=days)

        return project_events[
            (project_events["event_dt"] > start)
            &
            (project_events["event_dt"] <= t)
        ]

    last_7 = get_window(7)
    last_14 = get_window(14)
    last_28 = get_window(28)

    previous_14_count = (
        len(last_28) - len(last_14)
    )

    recent_14_count = len(last_14)

    class_counts = last_14[
        "structural_class"
    ].value_counts()

    records.append({
        "instance_id": project,
        "week_number": cp.week_number,
        "checkpoint_time": t,

        "scope_changes_last_7d": len(last_7),
        "scope_changes_last_14d": len(last_14),
        "scope_changes_last_28d": len(last_28),

        "epic_child_additions_14d": int(
            class_counts.get("EPIC_CHILD_ADDED", 0)
        ),

        "epic_child_removals_14d": int(
            class_counts.get("EPIC_CHILD_REMOVED", 0)
        ),

        "epic_reassignments_14d": int(
            class_counts.get("EPIC_REASSIGNED", 0)
        ),

        "issue_reclassifications_14d": int(
            class_counts.get("ISSUE_RECLASSIFIED", 0)
        ),

        "scope_affected_issues_14d": (
            last_14["affected_issue"].nunique()
        ),

        "scope_change_log_growth": float(
            np.log1p(recent_14_count)
            -
            np.log1p(previous_14_count)
        )
    })

scope_features = pd.DataFrame(records)

print("Scope feature records:", len(scope_features))
print("Projects:", scope_features["instance_id"].nunique())

display(scope_features.head(10))

print("\nFEATURE DISTRIBUTION")

display(
    scope_features.drop(columns=KEYS).describe()
)

Scope feature records: 987
Projects: 30


,instance_id,week_number,checkpoint_time,scope_changes_last_7d,scope_changes_last_14d,scope_changes_last_28d,epic_child_additions_14d,epic_child_removals_14d,epic_reassignments_14d,issue_reclassifications_14d,scope_affected_issues_14d,scope_change_log_growth
0,AEROGEAR-I01,1,2018-01-24 05:30:01+00:00,12,12,12,6,1,1,1,8,2.564949
1,AEROGEAR-I01,2,2018-01-31 05:30:01+00:00,0,12,12,6,1,1,1,8,2.564949
2,AEROGEAR-I01,3,2018-02-07 05:30:01+00:00,0,0,12,0,0,0,0,0,-2.564949
3,AEROGEAR-I01,4,2018-02-14 05:30:01+00:00,2,2,14,0,0,0,0,2,-1.466337
4,AEROGEAR-I01,5,2018-02-21 05:30:01+00:00,0,2,2,0,0,0,0,2,1.098612
5,AEROGEAR-I01,6,2018-02-28 05:30:01+00:00,1,1,3,0,0,0,0,1,-0.405465
6,AEROGEAR-I01,7,2018-03-07 05:30:01+00:00,16,17,19,8,0,0,0,17,1.791759
7,AEROGEAR-I01,8,2018-03-14 05:30:01+00:00,4,20,21,8,0,0,0,19,2.351375
8,AEROGEAR-I01,9,2018-03-21 05:30:01+00:00,0,4,21,0,0,0,0,3,-1.280934
9,AEROGEAR-I01,10,2018-03-28 05:30:01+00:00,0,0,20,0,0,0,0,0,-3.044522



FEATURE DISTRIBUTION


,scope_changes_last_7d,scope_changes_last_14d,scope_changes_last_28d,epic_child_additions_14d,epic_child_removals_14d,epic_reassignments_14d,issue_reclassifications_14d,scope_affected_issues_14d,scope_change_log_growth
count,987.000000,987.000000,987.000000,987.000000,987.000000,987.000000,987.000000,987.000000,987.000000
mean,0.877406,1.738602,3.422492,0.681864,0.072948,0.143870,0.287741,1.528875,0.026384
std,2.506010,3.867554,6.216906,2.753958,0.609881,0.687154,0.706591,3.480448,0.860389
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-3.637586
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-0.287682
50%,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,1.000000,2.000000,4.000000,0.000000,0.000000,0.000000,0.000000,2.000000,0.405465
max,37.000000,37.000000,59.000000,37.000000,10.000000,11.000000,6.000000,37.000000,3.637586


## 7. Consolidate feature groups

Combines workflow, temporal, dependency, KG, and scope features by checkpoint.

In [ ]:
# ============================================================
# NOTEBOOK 2 — STEP 7.1
# CONSOLIDATE ALL FEATURE GROUPS
# ============================================================

import pandas as pd
import numpy as np

KEYS = [
    "instance_id",
    "week_number",
    "checkpoint_time"
]

feature_groups = {
    "workflow": workflow_features,
    "temporal": temporal_features,
    "dependency": dependency_features,
    "structural_kg": kg_features,
    "scope": scope_features
}

combined_features = checkpoints_df[KEYS].copy()

for name, df in feature_groups.items():

    assert not df.duplicated(KEYS).any(), (
        f"Duplicate checkpoints in {name}"
    )

    new_columns = [
        c for c in df.columns
        if c not in KEYS
    ]

    duplicates = set(new_columns).intersection(
        combined_features.columns
    )

    assert not duplicates, (
        f"Duplicate feature names in {name}: {duplicates}"
    )

    combined_features = combined_features.merge(
        df[KEYS + new_columns],
        on=KEYS,
        how="left",
        validate="one_to_one"
    )

print("Combined feature table:", combined_features.shape)

print("Projects:", combined_features["instance_id"].nunique())

print("Feature count:", len(combined_features.columns) - len(KEYS))

display(combined_features.head())

Combined feature table: (987, 71)
Projects: 30
Feature count: 68


,instance_id,week_number,checkpoint_time,total_issues,open_issues,reopened_issues,blocked_status_issues,unknown_status_issues,recent_status_changes_7d,recent_status_changes_14d,...,kg_edge_growth_14d,scope_changes_last_7d,scope_changes_last_14d,scope_changes_last_28d,epic_child_additions_14d,epic_child_removals_14d,epic_reassignments_14d,issue_reclassifications_14d,scope_affected_issues_14d,scope_change_log_growth
0,AEROGEAR-I01,1,2018-01-24 05:30:01+00:00,162,4,2,0,14,2,2,...,71,12,12,12,6,1,1,1,8,2.564949
1,AEROGEAR-I01,2,2018-01-31 05:30:01+00:00,162,7,2,0,11,3,5,...,100,0,12,12,6,1,1,1,8,2.564949
2,AEROGEAR-I01,3,2018-02-07 05:30:01+00:00,162,9,2,0,9,2,5,...,43,0,0,12,0,0,0,0,0,-2.564949
3,AEROGEAR-I01,4,2018-02-14 05:30:01+00:00,164,7,2,0,11,3,3,...,53,2,2,14,0,0,0,0,2,-1.466337
4,AEROGEAR-I01,5,2018-02-21 05:30:01+00:00,166,6,2,0,11,6,8,...,73,0,2,2,0,0,0,0,2,1.098612


## 8. Select 34 predictive features

Retains a compact, domain-driven numerical feature set.

In [ ]:
# ============================================================
# NOTEBOOK 2 — STEP 8.1
# DOMAIN-DRIVEN FEATURE SELECTION
# ============================================================

import pandas as pd
import numpy as np

KEYS = [
    "instance_id",
    "week_number",
    "checkpoint_time"
]

SELECTED_FEATURES = [

    # Historical Jira workflow
    "total_issues",
    "open_issue_ratio",
    "reopened_issues",
    "blocked_status_issues",
    "unknown_status_ratio",
    "recent_status_changes_14d_bulk_adjusted",
    "status_change_ratio_14d_bulk_adjusted",

    # Temporal activity
    "log_events_last_7d",
    "log_events_last_28d",
    "event_log_growth_bulk_adjusted",
    "event_volatility_bulk_adjusted",
    "active_event_days_14d_bulk_adjusted",
    "new_issues_last_14d",
    "project_age_days",

    # Dependency graph
    "dependency_links",
    "blocked_issue_count",
    "max_blocking_degree",
    "max_blocker_count",
    "dependency_components",
    "max_dependency_chain",
    "dependency_graph_density",
    "dependency_node_coverage",

    # Structural Temporal KG
    "kg_entity_nodes",
    "kg_active_entities_14d",
    "kg_max_entity_degree",
    "kg_entity_concentration",
    "kg_new_entities_14d",

    # Historical scope changes
    "scope_changes_last_14d",
    "scope_changes_last_28d",
    "epic_child_additions_14d",
    "epic_reassignments_14d",
    "issue_reclassifications_14d",
    "scope_affected_issues_14d",
    "scope_change_log_growth"
]

missing = [
    c for c in SELECTED_FEATURES
    if c not in combined_features.columns
]

assert not missing, f"Missing features: {missing}"

selected_features_df = combined_features[
    KEYS + SELECTED_FEATURES
].copy()

print("Selected features:", len(SELECTED_FEATURES))
print("Feature matrix:", selected_features_df.shape)
print("Projects:", selected_features_df["instance_id"].nunique())

display(selected_features_df.head())

Selected features: 34
Feature matrix: (987, 37)
Projects: 30


,instance_id,week_number,checkpoint_time,total_issues,open_issue_ratio,reopened_issues,blocked_status_issues,unknown_status_ratio,recent_status_changes_14d_bulk_adjusted,status_change_ratio_14d_bulk_adjusted,...,kg_max_entity_degree,kg_entity_concentration,kg_new_entities_14d,scope_changes_last_14d,scope_changes_last_28d,epic_child_additions_14d,epic_reassignments_14d,issue_reclassifications_14d,scope_affected_issues_14d,scope_change_log_growth
0,AEROGEAR-I01,1,2018-01-24 05:30:01+00:00,162,0.024691,2,0,0.086420,2,0.012346,...,42,0.591549,6,12,12,6,1,1,8,2.564949
1,AEROGEAR-I01,2,2018-01-31 05:30:01+00:00,162,0.043210,2,0,0.067901,5,0.030864,...,42,0.415842,7,12,12,6,1,1,8,2.564949
2,AEROGEAR-I01,3,2018-02-07 05:30:01+00:00,162,0.055556,2,0,0.055556,5,0.030864,...,42,0.368421,2,0,12,0,0,0,0,-2.564949
3,AEROGEAR-I01,4,2018-02-14 05:30:01+00:00,164,0.042683,2,0,0.067073,3,0.018293,...,42,0.272727,2,2,14,0,0,0,2,-1.466337
4,AEROGEAR-I01,5,2018-02-21 05:30:01+00:00,166,0.036145,2,0,0.066265,8,0.048193,...,42,0.224599,4,2,2,0,0,0,2,1.098612


## 9. Integrate three ground-truth labels

Adds Schedule, Scope, and Dependency Risk labels without future outcome columns.

In [ ]:
# Deterministically discover label files for the existing integration routine.
LABEL_DIR = ROOT / "Temporal_snapshots"
label_files = sorted(
    p for p in LABEL_DIR.rglob("*")
    if p.is_file()
    and p.suffix.lower() in {".csv", ".parquet"}
    and "label" in p.name.lower()
)
assert label_files, "No label files found under Temporal_snapshots"

# ============================================================
# NOTEBOOK 2 — STEP 9.2
# INTEGRATE GROUND-TRUTH LABELS
# ============================================================

KEYS = [
    "instance_id",
    "week_number",
    "checkpoint_time"
]

TARGETS = [
    "schedule_risk_label",
    "scope_risk_label",
    "dependency_risk_label"
]

final_dataset = selected_features_df.copy()

for target in TARGETS:

    candidates = []

    for p in label_files:
        try:
            df = (
                pd.read_parquet(p)
                if p.suffix == ".parquet"
                else pd.read_csv(p)
            )
        except Exception:
            continue

        if target in df.columns and all(
            c in df.columns for c in KEYS
        ):
            candidates.append((p, df))

    if not candidates:
        raise FileNotFoundError(
            f"No checkpoint-aligned file found for {target}"
        )

    # Prefer the dedicated Labels directory
    candidates.sort(
        key=lambda x: (
            "Labels" not in x[0].parts,
            str(x[0])
        )
    )

    path, labels = candidates[0]

    labels = labels[KEYS + [target]].copy()

    labels["checkpoint_time"] = pd.to_datetime(
        labels["checkpoint_time"],
        utc=True,
        format="mixed"
    )

    assert not labels.duplicated(KEYS).any(), (
        f"Duplicate checkpoints in {target}"
    )

    assert len(labels) == 987, (
        f"Expected 987 rows for {target}, got {len(labels)}"
    )

    final_dataset = final_dataset.merge(
        labels,
        on=KEYS,
        how="left",
        validate="one_to_one"
    )

    print(f"Loaded {target} from {path.name}")

print("\nFINAL DATASET SHAPE:", final_dataset.shape)

display(final_dataset.head())

print("\nLABEL DISTRIBUTIONS")

for target in TARGETS:
    print("\n", target)
    print(final_dataset[target].value_counts(dropna=False))

Loaded schedule_risk_label from phase2_partial_risk_labels.csv
Loaded scope_risk_label from phase2_three_risk_labels.csv
Loaded dependency_risk_label from dependency_escalation_labels.csv

FINAL DATASET SHAPE: (987, 40)


,instance_id,week_number,checkpoint_time,total_issues,open_issue_ratio,reopened_issues,blocked_status_issues,unknown_status_ratio,recent_status_changes_14d_bulk_adjusted,status_change_ratio_14d_bulk_adjusted,...,scope_changes_last_14d,scope_changes_last_28d,epic_child_additions_14d,epic_reassignments_14d,issue_reclassifications_14d,scope_affected_issues_14d,scope_change_log_growth,schedule_risk_label,scope_risk_label,dependency_risk_label
0,AEROGEAR-I01,1,2018-01-24 05:30:01+00:00,162,0.024691,2,0,0.086420,2,0.012346,...,12,12,6,1,1,8,2.564949,0,0,0
1,AEROGEAR-I01,2,2018-01-31 05:30:01+00:00,162,0.043210,2,0,0.067901,5,0.030864,...,12,12,6,1,1,8,2.564949,0,0,0
2,AEROGEAR-I01,3,2018-02-07 05:30:01+00:00,162,0.055556,2,0,0.055556,5,0.030864,...,0,12,0,0,0,0,-2.564949,1,0,0
3,AEROGEAR-I01,4,2018-02-14 05:30:01+00:00,164,0.042683,2,0,0.067073,3,0.018293,...,2,14,0,0,0,2,-1.466337,1,0,0
4,AEROGEAR-I01,5,2018-02-21 05:30:01+00:00,166,0.036145,2,0,0.066265,8,0.048193,...,2,2,0,0,0,2,1.098612,1,1,1



LABEL DISTRIBUTIONS

 schedule_risk_label
schedule_risk_label
0    722
1    265
Name: count, dtype: int64

 scope_risk_label
scope_risk_label
0    820
1    167
Name: count, dtype: int64

 dependency_risk_label
dependency_risk_label
0    899
1     88
Name: count, dtype: int64


## 9.2 Validate final labelled dataset

Checks feature integrity, target completeness, and expected label distributions.

In [ ]:
# ============================================================
# NOTEBOOK 2 — STEP 9.3
# FINAL LABELLED DATASET VALIDATION
# ============================================================

import numpy as np

checks = {}

checks["987 checkpoints"] = len(final_dataset) == 987

checks["30 projects"] = (
    final_dataset["instance_id"].nunique() == 30
)

checks["No duplicate checkpoints"] = (
    not final_dataset.duplicated(KEYS).any()
)

checks["34 predictive features"] = (
    len(SELECTED_FEATURES) == 34
)

checks["All labels present"] = (
    final_dataset[TARGETS].notna().all().all()
)

checks["All labels binary"] = (
    final_dataset[TARGETS].isin([0, 1]).all().all()
)

checks["No missing predictive features"] = (
    final_dataset[SELECTED_FEATURES].notna().all().all()
)

checks["All predictive features finite"] = (
    np.isfinite(
        final_dataset[SELECTED_FEATURES]
        .to_numpy(dtype=float)
    ).all()
)

checks["Dependency labels match Notebook 1"] = (
    final_dataset["dependency_risk_label"].sum() == 88
)

checks["Schedule labels match Notebook 1"] = (
    final_dataset["schedule_risk_label"].sum() == 265
)

checks["No Decision Risk target"] = (
    "decision_risk_label" not in final_dataset.columns
)

validation_df = pd.DataFrame([
    {
        "validation": name,
        "status": "PASS" if passed else "FAIL"
    }
    for name, passed in checks.items()
])

display(validation_df)

assert all(checks.values()), (
    "Final labelled dataset validation failed."
)

print("\nALL FINAL DATASET VALIDATIONS PASSED")

,validation,status
0,987 checkpoints,PASS
1,30 projects,PASS
2,No duplicate checkpoints,PASS
3,34 predictive features,PASS
4,All labels present,PASS
5,All labels binary,PASS
6,No missing predictive features,PASS
7,All predictive features finite,PASS
8,Dependency labels match Notebook 1,PASS
9,Schedule labels match Notebook 1,PASS



ALL FINAL DATASET VALIDATIONS PASSED


## 9.3 Verify labels against consolidated ground truth

Compares all three risk labels row-by-row before exporting.

In [ ]:
# ============================================================
# FINAL CROSS-CHECK AGAINST CONSOLIDATED GROUND TRUTH
# ============================================================

from pathlib import Path
import pandas as pd

path = (
    Path("/content/drive/MyDrive/GC1")
    / "Temporal_snapshots/Labels/phase2_three_risk_labels.csv"
)

reference = pd.read_csv(path)

KEYS = ["instance_id", "week_number", "checkpoint_time"]

TARGETS = [
    "schedule_risk_label",
    "scope_risk_label",
    "dependency_risk_label"
]

reference["checkpoint_time"] = pd.to_datetime(
    reference["checkpoint_time"], utc=True
)

comparison = final_dataset[KEYS + TARGETS].merge(
    reference[KEYS + TARGETS],
    on=KEYS,
    how="outer",
    validate="one_to_one",
    indicator=True,
    suffixes=("_export", "_reference")
)

assert (comparison["_merge"] == "both").all()

for target in TARGETS:
    assert (
        comparison[f"{target}_export"]
        == comparison[f"{target}_reference"]
    ).all(), f"Mismatch detected: {target}"

print("ALL THREE RISK LABELS MATCH THE CONSOLIDATED FILE")
print("Verified checkpoints:", len(comparison))

ALL THREE RISK LABELS MATCH THE CONSOLIDATED FILE
Verified checkpoints: 987


## 10. Export TabPFN-ready datasets

Saves the labelled matrix, feature-only matrix, feature list, and validation report.

In [ ]:
assert (validation_df["status"] == "PASS").all(), "Refusing to export invalid data"

# ============================================================
# NOTEBOOK 2 — STEP 9.4
# EXPORT TABPFN-READY FEATURE DATASET
# ============================================================

OUTPUT_DIR = (
    ROOT / "Temporal_snapshots"
    / "feature_engineering_output"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Full labelled dataset
final_dataset.to_parquet(
    OUTPUT_DIR / "tabpfn_labelled_dataset.parquet",
    index=False
)

# Predictive features only
selected_features_df.to_parquet(
    OUTPUT_DIR / "predictive_features.parquet",
    index=False
)

# Feature list
pd.DataFrame({
    "feature_name": SELECTED_FEATURES
}).to_csv(
    OUTPUT_DIR / "selected_feature_list.csv",
    index=False
)

# Validation report
validation_df.to_csv(
    OUTPUT_DIR / "feature_validation_report.csv",
    index=False
)

print("FINAL FEATURE DATASET EXPORTED")
print("Output:", OUTPUT_DIR)
print("Records:", len(final_dataset))
print("Features:", len(SELECTED_FEATURES))
print("Risk labels:", len(TARGETS))

FINAL FEATURE DATASET EXPORTED
Output: /content/drive/MyDrive/GC1/Temporal_snapshots/feature_engineering_output
Records: 987
Features: 34
Risk labels: 3
